# ツール関数と MCP サーバーの利用方法

このノートブックでは、ファンクションコールで外部の API サービスを呼び出す AI エージェントの作成方法を学びます。

ツール関数を独自に実装する方法と、既存の MCP サーバーを利用する方法をそれぞれ説明します。

## 事前準備

**[TFM-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。ここでは、特に、MCP ToolSet で必要なパッケージを追加しています。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk[mcp]==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[TFM-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[TMF-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[TMF-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[TMF-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os, requests
from typing import Dict, Any
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.colab import userdata

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

/root/.local/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


**[TMF-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        events = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            events.append(event)
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result), events

## ツール関数を独自に実装する例

**[TMF-07]**

Open-Meteo API を用いて、指定した都道府県の天気予報データを取得する関数 `get_weather_forecast` を定義します。

In [16]:
JSON_SCHEMA = '''
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "OpenMeteoForecastResponse",
  "type": "object",
  "required": [
    "latitude",
    "longitude",
    "generationtime_ms",
    "utc_offset_seconds",
    "timezone",
    "timezone_abbreviation",
    "elevation",
    "hourly_units",
    "hourly"
  ],
  "properties": {
    "latitude": {
      "type": "number",
      "description": "APIが算出した対象地点の緯度（小数点以下桁数はグリッド解像度に依存）"
    },
    "longitude": {
      "type": "number",
      "description": "APIが算出した対象地点の経度"
    },
    "generationtime_ms": {
      "type": "number",
      "description": "サーバー側でのレスポンス生成時間（ミリ秒）"
    },
    "utc_offset_seconds": {
      "type": "integer",
      "description": "UTC（協定世界時）からの時差（秒単位）。日本（JST）の場合は 32400"
    },
    "timezone": {
      "type": "string",
      "description": "対象地域のタイムゾーン名（例: 'Asia/Tokyo'）"
    },
    "timezone_abbreviation": {
      "type": "string",
      "description": "タイムゾーンの略称（例: 'JST' や 'GMT+9'）"
    },
    "elevation": {
      "type": "number",
      "description": "対象地点の標高（メートル）"
    },
    "hourly_units": {
      "type": "object",
      "description": "hourly オブジェクトに含まれる各データの表示単位",
      "required": [
        "time",
        "temperature_2m",
        "precipitation_probability",
        "rain",
        "weather_code"
      ],
      "properties": {
        "time": { "type": "string", "enum": ["iso8601"] },
        "temperature_2m": { "type": "string", "enum": ["°C", "°F"] },
        "precipitation_probability": { "type": "string", "enum": ["%"] },
        "rain": { "type": "string", "enum": ["mm", "inch"] },
        "weather_code": { "type": "string", "enum": ["wmo code"] }
      }
    },
    "hourly": {
      "type": "object",
      "description": "1時間ごとの時系列予報データ。配列要素に各時間のデータが紐付きます（通常168要素 = 7日分）。",
      "required": [
        "time",
        "temperature_2m",
        "precipitation_probability",
        "rain",
        "weather_code"
      ],
      "properties": {
        "time": {
          "type": "array",
          "items": { "type": "string", "format": "date-time" },
          "description": "ISO8601形式のDateTime文字列配列（例: '2026-09-02T00:00'）"
        },
        "temperature_2m": {
          "type": "array",
          "items": { "type": "number" },
          "description": "地上2mの気温の配列"
        },
        "precipitation_probability": {
          "type": "array",
          "items": { "type": "integer", "minimum": 0, "maximum": 100 },
          "description": "降水確率（0〜100%）の配列"
        },
        "rain": {
          "type": "array",
          "items": { "type": "number", "minimum": 0 },
          "description": "1時間あたりの雨量（mm）の配列"
        },
        "weather_code": {
          "type": "array",
          "items": { "type": "integer" },
          "description": "WMO天候コード（0:快晴, 1〜3:晴れ〜曇り, 51〜67:雨, 95〜99:雷雨 など）の配列"
        }
      }
    }
  }
}
'''

async def get_weather_forecast(prefecture_name: str) -> Dict[str, Any]:
    f"""
    指定された都道府県（県庁所在地）の当日から始まる7日間（168時間）の天気予報を取得します。

    Args:
        prefecture_name (str): 都道府県名
        - '東京都', '京都府', '鹿児島県' のみが利用可能

    Returns:
        Dict[str, Any]: Open-Meteo APIからの JSON レスポンスデータ

    Raises:
        ValueError: サポートされていない都道府県名が指定された場合
        requests.RequestException: API リクエストに失敗した場合

    JSON レスポンスデータのスキーマ:
    {JSON_SCHEMA}
    """

    prefecture_coordinates = {
        '東京都': {'lat': 35.6895, 'lon': 139.6917},
        '京都府': {'lat': 35.0211, 'lon': 135.7556},
        '鹿児島県': {'lat': 31.5602, 'lon': 130.5581},
    }
    if prefecture_name not in prefecture_coordinates:
        raise ValueError(f'{prefecture_name} はサポートされていません。')
    coords = prefecture_coordinates[prefecture_name]

    # Open-Meteo API エンドポイント
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': coords['lat'],
        'longitude': coords['lon'],
        'hourly': 'temperature_2m,precipitation_probability,rain,weather_code',
        'timezone': 'auto'
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()

**[TMF-08]**

関数 `get_weather_forecast` をツール関数に持つ LlmAgent オブジェクトと AdkApp オブジェクトを作成します。

In [17]:
instruction = '''
あなたは気象予報データに基づいて、ユーザーの質問に回答するエージェントです。
- get_weather_forecast で気象予報データを取得します。
- 取得したデータに基づいて、客観的な情報を提供してください。
'''

weather_forecast_agent = LlmAgent(
    name='weather_forecast_agent',
    model='gemini-3.5-flash-lite',
    description='気象予報データに基づいて質問に回答するエージェント',
    instruction=instruction,
    tools=[get_weather_forecast],
)

weather_forecast_app = AdkApp(
    agent=weather_forecast_agent,
    app_name='weather_forecast_app',
)

**[TMF-09]**

作成したAIエージェントと会話します。

In [18]:
chat_client = ChatClient(weather_forecast_app)

query = '''
明日から京都旅行です。雨具の用意は必要ですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


明日からの京都旅行ですね！京都の天気予報を確認したところ、**雨具の用意をしてお出かけいただくことを強くおすすめします。**

明日（9月4日）の京都の天気予報は以下の通りです：

* **降水確率：** 深夜から一日を通して高めですが、特に**朝3時頃〜夜にかけて40%〜90%台**と高くなります。
* **雨の予想：** 日中を通じて断続的に雨が降る予報となっており、時間帯によってはまとまった雨（1〜4mm程度/h）になる可能性もあります。

折りたたみ傘やレインコートなどの雨具をバッグに入れておくと安心です。
楽しいご旅行になりますように！

**[TMF-10]**

AIエージェントからの応答イベントを確認します。ファンクションコールで `get_weather_forcast` が使用されたことがわかります。

In [19]:
for event in events:
    print(event['content'])

{'parts': [{'function_call': {'id': 'call_3460054', 'args': {'prefecture_name': '京都府'}, 'name': 'get_weather_forecast'}, 'thought_signature': 'AY89a18H_h2vYf17UzJfZm7GHW8_MC1F-ApW6azcEEE2sa3Yexz-NPkmNCmYi0ASFtUq1HNdJkXRuNZiufiKzjn9jw6YQHRrFH_eEO9P3wAjfxg='}], 'role': 'model'}
{'parts': [{'function_response': {'id': 'call_3460054', 'name': 'get_weather_forecast', 'response': {'latitude': 35.0, 'longitude': 135.75, 'generationtime_ms': 0.2897977828979492, 'utc_offset_seconds': 32400, 'timezone': 'Asia/Tokyo', 'timezone_abbreviation': 'GMT+9', 'elevation': 56.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation_probability': '%', 'rain': 'mm', 'weather_code': 'wmo code'}, 'hourly': {'time': ['2026-09-03T00:00', '2026-09-03T01:00', '2026-09-03T02:00', '2026-09-03T03:00', '2026-09-03T04:00', '2026-09-03T05:00', '2026-09-03T06:00', '2026-09-03T07:00', '2026-09-03T08:00', '2026-09-03T09:00', '2026-09-03T10:00', '2026-09-03T11:00', '2026-09-03T12:00', '2026-09-03T13:00

**[TMF-11]**

関数 `get_weather_forcast` の docstring に記載した「当日から始まる7日間（168時間）の天気予報」という内容を理解しているか確認します。

In [20]:
query = '''
なぜ明日が9月4日だとわかったのですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

取得した気象予報データの中に含まれている時刻（`time`）のデータを確認したためです。

データ内の最初のタイムスタンプが「`2026-09-03T00:00`」となっており、それに続く時間経過のデータから、翌日のデータが「`2026-09-04`」に該当することが確認できたため、明日が9月4日であると判断いたしました。

## MCP サーバーを利用する例

事前準備として、クラウドコンソールの「Google Maps Platform」→「鍵と認証情報」で API キーを取得して、Colaboratory のシークレットに `google_maps_api_key` という名前で保存しておきます。

**[TMF-11]**

Google Maps Grounding Lite MCP の公開 MCP サーバーを利用するツールセットを定義します。

In [45]:
# Google Maps Grounding Lite MCP エンドポイント
MAPS_MCP_ENDPOINT = "https://mapstools.googleapis.com/mcp"

google_maps_toolset = McpToolset(
        connection_params=StreamableHTTPServerParams(
            url=MAPS_MCP_ENDPOINT,
            headers={'X-Goog-Api-Key': userdata.get('google_maps_api_key')},
        ),
        tool_filter=['search_places', 'compute_routes'] # 'lookup_weather'
    )

**[TMF-12]**

定義したツールセットをツール関数に持つ LlmAgent オブジェクトと AdkApp オブジェクトを作成します。

In [46]:
instruction='''
あなたは Google Maps を活用して施設情報を提供するエージェントです。
- Google Maps ツールを使用して得られた最新情報に基づいて回答してください。
- 可能な場合は Google Maps リンクを提供してください。
'''

facility_information_agent = LlmAgent(
    name='facility_information_agent',
    model='gemini-3.5-flash-lite',
    description='Google Maps に基づいて施設情報を提供するエージェント',
    instruction=instruction,
    tools=[google_maps_toolset],
)

facility_information_app = AdkApp(
    agent=facility_information_agent,
    app_name='facility_information_app',
)

**[TMF-13]**

作成したAIエージェントと会話します。MCP サーバーから取得したツールの機能を理解していることを確認します。

In [63]:
chat_client = ChatClient(facility_information_app)

query = '''
何ができますか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

私は Google Maps を活用して、以下のような施設情報や移動に関するお手伝いができるエージェントです：

1. **施設の検索と詳細情報**
   - 周辺のレストラン、カフェ、観光スポット、公園、公共施設などを探します。
   - 施設の住所、営業時間、連絡先、レビュー、Google マップへのリンクなどをご案内できます。

2. **ルートの検索**
   - ある地点から別の地点までの移動ルート（車や徒歩）や、所要時間、距離を計算します。

何かお探しのお店や場所、行きたいルートなどはありますか？お気軽にご相談ください！

**[TMF-14]**

具体的な施設の検索を行ってみます。

In [64]:
query = '''
渋谷駅の近くにある郵便局を教えて。特にハチ公前出口から最も近いのはどれですか？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

渋谷駅周辺にある主な郵便局と、ハチ公前出口からの距離・所要時間の比較をご案内します。

ハチ公前出口から最も近い郵便局は **「渋谷中央街郵便局」** です！

---

### 1. 渋谷中央街郵便局（最も近い局）
* **ハチ公前出口からの距離・所要時間**: 約 417m / 徒歩約 6分
* **住所**: 東京都渋谷区道玄坂1丁目10-2 渋谷Crビル 1F
* **営業時間**: 月〜金 9:00〜17:00（土日祝休み）
* [Google マップで見る](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b57b27769c5:0xd42316e784b9ef7)

### 2. 渋谷郵便局（本局・比較的大きめの局）
* **ハチ公前出口からの距離・所要時間**: 約 422m / 徒歩約 7分
* **住所**: 東京都渋谷区渋谷1-12-13
* **営業時間**: 平日 9:00〜21:00、土日 9:00〜18:00
* [Google マップで見る](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e)
*(※窓口の営業時間や対応時間が長めで、ゆうちょ銀行渋谷店も併設されています)*

### 3. 渋谷神南郵便局
* **ハチ公前出口からの距離・所要時間**: 約 508m / 徒歩約 7分
* **住所**: 東京都渋谷区神南1-21-1 日本生命渋谷ビル
* **営業時間**: 月〜金 9:00〜17:00（土日祝休み）
* [Google マップで見る](https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188ca8882f21bb:0xe41fdf24841341b4)

---

※徒歩での移動には、歩道や混雑状況によって多少時間が前後する場合がありますのでご注意ください。週末の利用や夜間の利用を希望される場合は、営業時間が長い「渋谷郵便局」が便利です。

**[TMF-15]**

AIエージェントからの応答イベントを確認します。この例では、ファンクションコールが複数回実行されており、`search_places` と `compute_routes` のツールが使用されています。

In [65]:
for event in events:
    print(event['content'])

{'parts': [{'function_call': {'id': 'call_3056338', 'args': {'textQuery': '渋谷駅 郵便局'}, 'name': 'search_places'}, 'thought_signature': 'AY89a1_vL0PWd_q2qIX0Ik-FTHrkZdQpTfWO42k-OaYK4ahpkg2FkpeRHS_J7aMzV9NnRAyExVtt9jh3aJvg4A7jVZB0RTtxkXaoGy2nb0F9Lro='}], 'role': 'model'}
{'parts': [{'function_response': {'id': 'call_3056338', 'name': 'search_places', 'response': {'content': [{'type': 'text', 'text': '{\n  "places": [\n    {\n      "place": "places/ChIJG8fUi1iLGGARTi54pg9fvE4",\n      "id": "ChIJG8fUi1iLGGARTi54pg9fvE4",\n      "location": {\n        "latitude": 35.6599551,\n        "longitude": 139.7041647\n      },\n      "googleMapsLinks": {\n        "directionsUrl": "https://www.google.com/maps/dir//\'\'/data=!4m7!4m6!1m1!4e2!1m2!1m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e!3e0",\n        "placeUrl": "https://www.google.com/maps/place//data=!4m2!3m1!1s0x60188b588bd4c71b:0x4ebc5f0fa6782e4e",\n        "writeAReviewUrl": "https://www.google.com/maps/place//data=!4m3!3m2!1s0x60188b588bd4c71b

**[TMF-16]**

どのようにして最も近い郵便局を検索したのか確認します。

In [66]:
query = '''
なぜ渋谷中央街郵便局が最も近いとわかったの？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

渋谷駅のハチ公前出口と、周辺にある複数の郵便局（渋谷中央街郵便局、渋谷郵便局、渋谷神南郵便局など）のそれぞれの位置情報を Google Maps を使って特定し、**「徒歩ルートの検索（Compute Routes）」**によって実際に歩いた場合の距離と所要時間を計算して比較したためです。

具体的には以下の手順で確認しました：

1. **位置の特定**
   - 渋谷駅の「ハチ公前出口（バス停・エリア）」の位置を検索しました。
   - 周辺の郵便局（渋谷中央街郵便局、渋谷郵便局、渋谷神南郵便局）の位置情報を取得しました。
2. **ルート計算（徒歩）**
   - ハチ公前出口から各郵便局までの「徒歩（WALK）」での経路をそれぞれ計算しました。
     - **渋谷中央街郵便局**までの距離：**約 417m**（徒歩約 6分）
     - **渋谷郵便局**までの距離：**約 422m**（徒歩約 7分）
     - **渋谷神南郵便局**までの距離：**約 508m**（徒歩約 7分）

直線距離ではなく、実際に歩く道のりで最も近かったため、「渋谷中央街郵便局が最も近い」とご案内できました！